# 01 - Exploración del Dataset

**Grupo 15 | UMSS FCyT | IA I/2026**

Objetivos:
- Cargar el dataset de encuestas estudiantiles
- Analizar estadísticas básicas del corpus
- Visualizar distribuciones de Likert, dimensiones y docentes
- Inspeccionar longitudes y calidad del texto

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Librerías cargadas ✅')

## 1. Cargar dataset

In [ ]:
df = pd.read_csv('../data/raw/encuesta_sintetica.csv')
print(f'Filas: {len(df)} | Columnas: {len(df.columns)}')
df.head()

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print('\nValores nulos:')
print(df.isnull().sum())

## 2. Estadísticas básicas

In [ ]:
print('=== Resumen del corpus ===')
print(f"Total comentarios : {len(df)}")
print(f"Docentes únicos   : {df['docente'].nunique()} → {df['docente'].unique().tolist()}")
print(f"Materias          : {df['materia'].nunique()} → {df['materia'].unique().tolist()}")
print(f"Dimensiones       : {df['dimension'].nunique()} → {df['dimension'].unique().tolist()}")
print(f"Likert (rango)    : {df['calificacion_likert'].min()} – {df['calificacion_likert'].max()}")
print(f"Likert (media)    : {df['calificacion_likert'].mean():.2f}")

## 3. Distribución de calificaciones Likert

In [ ]:
labels = {1: 'Muy malo', 2: 'Malo', 3: 'Regular', 4: 'Bueno', 5: 'Muy bueno'}
colors = ['#d32f2f', '#f57c00', '#ffd600', '#388e3c', '#1565c0']

counts = df['calificacion_likert'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar([labels[i] for i in counts.index], counts.values,
            color=[colors[i-1] for i in counts.index])
axes[0].set_title('Distribución de calificaciones Likert', fontweight='bold')
axes[0].set_ylabel('Respuestas')
axes[0].tick_params(axis='x', rotation=20)

axes[1].pie(counts.values, labels=[labels[i] for i in counts.index],
            colors=[colors[i-1] for i in counts.index],
            autopct='%1.0f%%', startangle=140)
axes[1].set_title('Proporción por calificación', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/fig_likert.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Comentarios por dimensión y docente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

df['dimension'].value_counts().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Comentarios por dimensión', fontweight='bold')
axes[0].set_xlabel('Cantidad')

df['docente'].value_counts().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Comentarios por docente', fontweight='bold')
axes[1].set_xlabel('Cantidad')

plt.tight_layout()
plt.show()

## 5. Heatmap: Likert promedio por docente × dimensión

In [ ]:
pivot = df.pivot_table(
    values='calificacion_likert',
    index='docente',
    columns='dimension',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn',
            vmin=1, vmax=5, linewidths=0.5, ax=ax)
ax.set_title('Calificación Likert promedio: Docente × Dimensión', fontweight='bold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('../data/processed/fig_heatmap_likert.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Longitud de comentarios

In [ ]:
df['num_chars'] = df['comentario'].astype(str).apply(len)
df['num_words'] = df['comentario'].astype(str).apply(lambda x: len(x.split()))

print(df[['num_chars', 'num_words']].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['num_words'], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Palabras por comentario', fontweight='bold')
axes[0].set_xlabel('Palabras')

axes[1].scatter(df['calificacion_likert'], df['num_words'], alpha=0.5, color='coral')
axes[1].set_title('Likert vs longitud del comentario', fontweight='bold')
axes[1].set_xlabel('Calificación Likert')
axes[1].set_ylabel('Palabras')

plt.tight_layout()
plt.show()

## 7. Top-30 palabras (sin preprocesar)

In [ ]:
todos_tokens = ' '.join(df['comentario'].astype(str)).lower().split()
top30 = Counter(todos_tokens).most_common(30)
words, freqs = zip(*top30)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(words)[::-1], list(freqs)[::-1], color='steelblue')
ax.set_title('Top-30 palabras más frecuentes (sin preprocesar)', fontweight='bold')
ax.set_xlabel('Frecuencia')
plt.tight_layout()
plt.show()

print('Nota: muchas son stopwords → necesitamos preprocesamiento (notebook 02).')

## 8. Guardar dataset con metadatos

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/encuesta_con_metadata.csv', index=False)
print('✅ Guardado en data/processed/encuesta_con_metadata.csv')